# SO-101: ghi dữ liệu, train ACT, đánh giá trên Google Colab

Notebook này làm đúng 3 bước như trên máy ở nhà (xem `docs/so_tay_ai.md`), chỉ khác là chạy trên GPU của Colab:

1. **Ghi dữ liệu**: policy viết tay chạy trong MuJoCo, ghi ảnh + góc khớp + lệnh (hoặc dùng dataset đã tải lên Drive).
2. **Train** ACT bằng `lerobot-train`.
3. **Đánh giá** trên bộ seed cố định, ghi replay để xem trên Studio ở máy nhà.

Kết quả lưu vào Google Drive (`MyDrive/vla/...`) để không mất khi Colab ngắt.

**Trước khi chạy:** menu *Runtime → Change runtime type → GPU* (Pro: chọn **A100** hoặc **L4**; T4 cũng được nhưng chậm hơn).
Chạy lần lượt từng ô từ trên xuống (Shift+Enter).

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Nối Google Drive

Mọi thứ cần giữ lại (dataset, checkpoint, kết quả) nằm trong `MyDrive/vla`. Colab sẽ hỏi quyền truy cập Drive: bấm cho phép.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
VLA = '/content/drive/MyDrive/vla'          # thư mục gốc trên Drive, giống D:/vla ở máy nhà
for sub in ('datasets', 'outputs/train', 'eval', 'replays'):
    os.makedirs(f'{VLA}/{sub}', exist_ok=True)
print('Drive sẵn sàng:', VLA)

## 3. Lấy code và cài thư viện

Code lấy từ GitHub (nhánh `my-work`). Thư mục phải tên `sim_engine` vì Python import theo tên đó.
`MUJOCO_GL=egl`: Colab không có màn hình, MuJoCo vẽ ảnh camera bằng GPU qua EGL.
Cài mất khoảng 3-5 phút. Nếu Colab báo *Restart session* sau khi cài thì bấm restart rồi chạy lại từ ô số 2.

In [ ]:
import os
os.environ['MUJOCO_GL'] = 'egl'
os.environ['HF_HOME'] = '/content/hf-cache'
# Khai báo driver EGL của NVIDIA: thiếu file này MuJoCo vẽ ảnh bằng CPU (llvmpipe), chậm ~160 lần (0,85 s mỗi ảnh)
p = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
if not os.path.exists(p):
    open(p, 'w').write('{"file_format_version" : "1.0.0", "ICD" : {"library_path" : "libEGL_nvidia.so.0"}}')
if os.path.isdir('/content/sim_engine/.git'):
    !git -C /content/sim_engine pull -q
else:
    !git clone -q -b my-work https://github.com/tvhhai/sim-engine.git /content/sim_engine
!git -C /content/sim_engine log --oneline -1
!pip install -q "lerobot[dataset,training]==0.7.0" "mujoco>=4.0.1" gymnasium opencv-python-headless h5py

In [ ]:
# Kiểm tra nhanh: dựng env, vẽ một ảnh camera
%cd /content
import os
os.environ['MUJOCO_GL'] = 'egl'
from sim_engine.learning import sim_setup
env, opt = sim_setup.make_env(3000)
obs, _ = env.reset(seed=3000, options=opt)
frame = sim_setup.to_frame(env, obs, [0] * 7, 'test', ('top', 'wrist'))
print({k: getattr(v, 'shape', v) for k, v in frame.items()})
env.close()

## 4. Chọn thí nghiệm

Mỗi thí nghiệm là một file trong `learning/configs/`. Đổi `EXP` và các thông số dưới đây cho khớp:

| EXP | CAMERAS | ENV_STATE | NOISE |
|---|---|---|---|
| `act_box4_v1` | `front,wrist` | False | 0 |
| `act_box4_topwrist` | `top,wrist` | False | 0 |
| `act_box4_wrist` | `wrist` | False | 0 |
| `act_box4_state` | (để trống) | True | 0 |
| `act_box4_topwrist_noise` | `top,wrist` | False | 0.02 |

`BATCH`: GPU Colab lớn hơn GTX 1650 nên tăng được (A100: 32-64, L4: 16-32, T4: 8-16). Đổi batch thì kết quả không còn so trực tiếp
với bản train ở nhà (batch 8); muốn so công bằng thì để 8.

In [ ]:
EXP = 'act_box4_topwrist'      # tên file config (không có .yaml)
CAMERAS = 'top,wrist'
ENV_STATE = False
NOISE = 0.0
EPISODES = 200                 # số ván thành công cần ghi
BATCH = 32
STEPS = 20000

DS_NAME = 'so101_box4_' + EXP.replace('act_box4_', '')        # vd so101_box4_topwrist
DS_LOCAL = f'/content/datasets/{DS_NAME}'                      # dataset đọc từ ổ của máy Colab (nhanh hơn đọc từ Drive)
DS_ZIP = f'{VLA}/datasets/{DS_NAME}.zip'                       # bản lưu trên Drive
OUT = f'{VLA}/outputs/train/{EXP}_colab_b{BATCH}'              # checkpoint lưu thẳng lên Drive
print(DS_LOCAL, DS_ZIP, OUT, sep='\n')

## 5. Có dataset

Ô dưới tự chọn:
- Đã có `MyDrive/vla/datasets/<tên>.zip` (ghi ở lần trước, hoặc bạn tải từ máy nhà lên) thì giải nén ra dùng.
- Chưa có thì ghi mới trên Colab (200 ván, khoảng 20-40 phút tuỳ máy), rồi nén lưu lên Drive cho lần sau.

Tải dataset từ máy nhà lên (không bắt buộc): nén thư mục `D:/vla/datasets/<tên>` thành `<tên>.zip`, kéo vào `MyDrive/vla/datasets/` trên drive.google.com.
Zip phải chứa thẳng các thư mục `data/ meta/ videos/` (không lồng thêm thư mục).

In [ ]:
import os, shutil, subprocess
if os.path.isdir(DS_LOCAL):
    print('Đã có dataset trên máy Colab:', DS_LOCAL)
elif os.path.exists(DS_ZIP):
    os.makedirs(DS_LOCAL, exist_ok=True)
    !unzip -q "{DS_ZIP}" -d "{DS_LOCAL}"
    print('Đã giải nén từ Drive:', DS_ZIP)
else:
    extra = (' --env-state' if ENV_STATE else '') + (f' --noise {NOISE}' if NOISE else '')
    !cd /content && python -m sim_engine.learning.record_dataset --episodes {EPISODES} --cameras "{CAMERAS}" {extra} --root {DS_LOCAL} --repo-id local/{DS_NAME} 2>&1 | grep -E "seed|Xong" | awk 'NR % 20 == 0 || /Xong/'
    shutil.make_archive(DS_ZIP[:-4], 'zip', DS_LOCAL)
    print('Đã ghi và lưu lên Drive:', DS_ZIP)
!du -sh {DS_LOCAL}

## 6. Train

In loss mỗi 100 bước; checkpoint mỗi 5000 bước vào `OUT` trên Drive.
Nếu Colab ngắt giữa chừng: chạy lại ô 2, 3, 4, 5 rồi chạy ô **6b (train tiếp)** thay vì ô này.

In [ ]:
!cd /content && python -m sim_engine.learning.train sim_engine/learning/configs/{EXP}.yaml --dataset.root={DS_LOCAL} --output_dir={OUT} --batch_size={BATCH} --steps={STEPS} --num_workers=4 2>&1 | grep -E --line-buffered "step:|Error|Traceback|error"

### 6b. Train tiếp sau khi bị ngắt

In [ ]:
!cd /content && lerobot-train --config_path={OUT}/checkpoints/last/pretrained_model/train_config.json --resume=true --dataset.root={DS_LOCAL} 2>&1 | grep -E --line-buffered "step:|Error|Traceback|error"

## 7. Đánh giá + replay

Chạy 100 seed của bộ đánh giá (giống ở nhà). Replay chép lên `MyDrive/vla/replays/`: tải file `.json` về đặt vào
`sim-engine/results/replays/` ở máy nhà, mở Studio → Replay → Nguồn replay.

In [ ]:
POLICY = f'{OUT}/checkpoints/last/pretrained_model'
NAME = f'{EXP}_colab'
!cd /content && python -m sim_engine.learning.eval_policy --policy {POLICY} --limit 100 --name {NAME} --out {VLA}/eval/{NAME}.json --replay {NAME} 2>&1 | grep -vE "Warning|warn"
!cp /content/sim_engine/results/replays/{NAME}.json {VLA}/replays/
print('Replay:', f'{VLA}/replays/{NAME}.json')

## 9. Cách khác: chạy cả bước 5-6-7 ở nền (để qua đêm)

Ô 5, 6, 7 chạy trong trang web: trang mất kết nối là hàng chờ ô bị mất. Ô dưới viết các bước đó thành một script và chạy **ở nền
trên máy Colab** (`nohup`), nên trang có mất kết nối thì máy vẫn làm tiếp. Chạy ô 1-4 trước (GPU, Drive, cài đặt, chọn thí nghiệm),
rồi chạy ô này **thay cho** ô 5-6-7. Xem tiến độ bằng ô kế tiếp (chạy lại bao nhiêu lần cũng được).

Lưu ý: Colab Pro vẫn tắt máy nếu không còn tab nào mở notebook quá lâu; cứ để tab mở.

In [ ]:
extra = (' --env-state' if ENV_STATE else '') + (f' --noise {NOISE}' if NOISE else '')
script = f'''set -x
export MUJOCO_GL=egl HF_HOME=/content/hf-cache
cd /content
if [ ! -d {DS_LOCAL} ]; then
  if [ -f {DS_ZIP} ]; then mkdir -p {DS_LOCAL} && unzip -q {DS_ZIP} -d {DS_LOCAL}
  else python -m sim_engine.learning.record_dataset --episodes {EPISODES} --cameras "{CAMERAS}"{extra} --root {DS_LOCAL} --repo-id local/{DS_NAME} && (cd {DS_LOCAL} && zip -qr {DS_ZIP} .)
  fi
fi
python -m sim_engine.learning.train sim_engine/learning/configs/{EXP}.yaml --dataset.root={DS_LOCAL} --output_dir={OUT} --batch_size={BATCH} --steps={STEPS} --num_workers=4
python -m sim_engine.learning.eval_policy --policy {OUT}/checkpoints/last/pretrained_model --limit 100 --name {EXP}_colab --out {VLA}/eval/{EXP}_colab.json --replay {EXP}_colab
cp /content/sim_engine/results/replays/{EXP}_colab.json {VLA}/replays/
echo ALL_DONE
'''
open('/content/run_all.sh', 'w').write(script)
!nohup bash /content/run_all.sh > /content/run.log 2>&1 &
print('Đã chạy nền. Xem tiến độ bằng ô dưới.')

In [ ]:
# Tiến độ: ghi dữ liệu (seed ...), train (step: ...), chấm điểm ([k/100] ...), xong hẳn khi thấy ALL_DONE
!tr '\r' '\n' < /content/run.log | grep -E "seed [0-9]+:|Xong|step:|\[[0-9]+/100\]|\"all\"|ALL_DONE|Error|Traceback" | tail -n 6
!cp /content/run.log {VLA}/{EXP}_colab_run.log

## 10. DAgger: AI lái, thầy sửa, train tiếp (chạy nền)

Dùng sau khi đã có một mạng train xong (vd `act_box4_topwrist_500`, 59/100). Chạy ô 1-4 trước (GPU, Drive, cài đặt; ô 4 chỉ để có
biến `VLA`), rồi chạy ô dưới. Các bước, tất cả ở nền như ô 9:

1. **Ghi DAgger** (`learning/dagger.py`): mỗi ván AI tự lái một đoạn ngẫu nhiên (15-250 bước), rồi thầy giáo (policy viết tay) tiếp quản
   từ đúng chỗ đó và làm nốt. Ghi đoạn thầy làm. Ván AI tự xong thì bỏ (không có gì để sửa). Lưu zip lên Drive.
2. **Gộp** dataset gốc + DAgger (`learning/merge_datasets.py`).
3. **Train tiếp** mạng cũ trên dataset gộp `FT_STEPS` bước (`learning/configs/act_box4_topwrist_500_dagger1.yaml`).
4. **Chấm** 100 seed, chép replay lên Drive.

Muốn thêm vòng 2: đổi `ROUND = 2`, `BASE_OUT` = thư mục output của vòng 1 (`FT_OUT` in ra ở ô dưới), chạy lại.

In [ ]:
ROUND = 1
BASE_OUT = f'{VLA}/outputs/train/act_box4_topwrist_500_colab_l4_b32'      # mạng hiện tại: AI lái khi ghi DAgger, và điểm xuất phát để train tiếp
BASE_DS = 'so101_box4_topwrist_500'                                        # dataset gốc (zip trên Drive)
DAG_N = 300                                                                # số ván thầy sửa cần ghi
FT_STEPS = 20000
DAG_DS = f'so101_box4_topwrist_dagger{ROUND}'
MERGED = f'so101_box4_topwrist_500_dagger{ROUND}'
FT_OUT = f'{VLA}/outputs/train/act_box4_topwrist_500_dagger{ROUND}_colab'
NAME = f'act_box4_topwrist_500_dagger{ROUND}_colab'
lines = [
    'set -x',
    'export MUJOCO_GL=egl HF_HOME=/content/hf-cache',
    'cd /content && mkdir -p /content/datasets',
    f'if [ ! -d /content/datasets/{BASE_DS} ]; then mkdir -p /content/datasets/{BASE_DS} && unzip -q {VLA}/datasets/{BASE_DS}.zip -d /content/datasets/{BASE_DS}; fi',
    f'if [ ! -d /content/datasets/{DAG_DS} ]; then',
    f'  if [ -f {VLA}/datasets/{DAG_DS}.zip ]; then mkdir -p /content/datasets/{DAG_DS} && unzip -q {VLA}/datasets/{DAG_DS}.zip -d /content/datasets/{DAG_DS}',
    f'  else python -m sim_engine.learning.dagger --policy {BASE_OUT}/checkpoints/last/pretrained_model --episodes {DAG_N} --root /content/datasets/{DAG_DS} --repo-id local/{DAG_DS} && (cd /content/datasets/{DAG_DS} && zip -qr {VLA}/datasets/{DAG_DS}.zip .)',
    '  fi',
    'fi',
    f'rm -rf /content/datasets/{MERGED}',
    f'python -m sim_engine.learning.merge_datasets --out /content/datasets/{MERGED} --repo-id local/{MERGED} /content/datasets/{BASE_DS} /content/datasets/{DAG_DS}',
    f'python -m sim_engine.learning.train sim_engine/learning/configs/act_box4_topwrist_500_dagger1.yaml --policy.path={BASE_OUT}/checkpoints/last/pretrained_model --dataset.repo_id=local/{MERGED} --dataset.root=/content/datasets/{MERGED} --output_dir={FT_OUT} --job_name={NAME} --steps={FT_STEPS} --num_workers=4',
    f'python -m sim_engine.learning.eval_policy --policy {FT_OUT}/checkpoints/last/pretrained_model --limit 100 --name {NAME} --out {VLA}/eval/{NAME}.json --replay {NAME}',
    f'cp /content/sim_engine/results/replays/{NAME}.json {VLA}/replays/',
    'echo ALL_DONE',
]
open('/content/run_dagger.sh', 'w').write('\n'.join(lines) + '\n')
!nohup bash /content/run_dagger.sh > /content/run.log 2>&1 &
print('Đã chạy nền DAgger vòng', ROUND, '| mạng mới sẽ ở', FT_OUT)

In [ ]:
# Tiến độ DAgger: ghi (seed ...: teacher_ok / ai_ok ...), gộp, train (step: ...), chấm ([k/100]); xong khi thấy ALL_DONE
!tr '\r' '\n' < /content/run.log | grep -E "seed [0-9]+: |Xong|gộp|step:|\[[0-9]+/100\]|\"all\"|ALL_DONE|Error|Traceback" | tail -n 6
!cp /content/run.log {VLA}/{NAME}_run.log

## 8. Đem checkpoint về máy nhà (khi cần)

Checkpoint đã nằm trên Drive: `MyDrive/vla/outputs/train/<EXP>_colab_b<BATCH>/checkpoints/last/pretrained_model/`.
Tải cả thư mục `pretrained_model` về (vd `D:/vla/outputs/train/...`) rồi chạy `eval_policy --policy <thư mục đó>` như bình thường.